# A/B Test Simulation: See the Waste

**Goal:** See why A/B testing wastes traffic when one option is clearly better.

**Time:** 15 minutes

In this notebook, you'll simulate an A/B test for two crude oil trading strategies and watch cumulative regret pile up as the test keeps allocating 50% of capital to the losing strategy, even after it's obviously inferior.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

## The Setup: Two Trading Strategies

You're testing two strategies for trading WTI crude oil futures:

- **Strategy A (Momentum):** 5% win rate (profitable on 5% of trades)
- **Strategy B (Mean Reversion):** 8% win rate (profitable on 8% of trades)

You don't know these true rates — you're running an A/B test to find out which is better.

**A/B test protocol:** Allocate exactly 50% of trades to each strategy until you reach statistical significance.

In [ ]:
# True win rates (unknown to the tester)
strategy_A_win_rate = 0.05
strategy_B_win_rate = 0.08

# Simulation parameters
n_trades = 10000
alpha = 0.05  # Significance level

print(f"True win rates:")
print(f"  Strategy A (Momentum):      {strategy_A_win_rate:.1%}")
print(f"  Strategy B (Mean Reversion): {strategy_B_win_rate:.1%}")
print(f"\nStrategy B is {(strategy_B_win_rate/strategy_A_win_rate - 1):.0%} better!")
print(f"\nBut the A/B test doesn't know this yet...")

## Run the A/B Test

We'll simulate 10,000 trades with strict 50/50 allocation.

In [ ]:
# Allocate trades: 50/50 split always
trades_A = []
trades_B = []
cumulative_regret = []
p_values = []

best_strategy_rate = max(strategy_A_win_rate, strategy_B_win_rate)

for t in range(n_trades):
    # A/B test: alternate between strategies (50/50)
    if t % 2 == 0:
        # Trade Strategy A
        outcome = np.random.rand() < strategy_A_win_rate
        trades_A.append(outcome)
        instant_regret = best_strategy_rate - strategy_A_win_rate
    else:
        # Trade Strategy B
        outcome = np.random.rand() < strategy_B_win_rate
        trades_B.append(outcome)
        instant_regret = best_strategy_rate - strategy_B_win_rate
    
    # Track cumulative regret
    if t == 0:
        cumulative_regret.append(instant_regret)
    else:
        cumulative_regret.append(cumulative_regret[-1] + instant_regret)
    
    # Calculate p-value (two-proportion z-test) every 100 trades
    if t % 100 == 99 and len(trades_A) > 10 and len(trades_B) > 10:
        n_A = len(trades_A)
        n_B = len(trades_B)
        wins_A = sum(trades_A)
        wins_B = sum(trades_B)
        
        p_A = wins_A / n_A
        p_B = wins_B / n_B
        p_pooled = (wins_A + wins_B) / (n_A + n_B)
        
        se = np.sqrt(p_pooled * (1 - p_pooled) * (1/n_A + 1/n_B))
        z_score = (p_B - p_A) / se if se > 0 else 0
        p_value = 2 * (1 - stats.norm.cdf(abs(z_score)))
        
        p_values.append((t, p_value))

print(f"A/B test completed: {n_trades} trades")
print(f"Strategy A trades: {len(trades_A)} (wins: {sum(trades_A)}, rate: {sum(trades_A)/len(trades_A):.1%})")
print(f"Strategy B trades: {len(trades_B)} (wins: {sum(trades_B)}, rate: {sum(trades_B)/len(trades_B):.1%})")
print(f"\nFinal cumulative regret: {cumulative_regret[-1]:.1f} missed wins")

## Visualize the Waste

This chart shows cumulative regret growing **linearly** over time. Every trade to Strategy A costs you ~3% in expected win rate.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: Cumulative regret
axes[0].plot(cumulative_regret, linewidth=2, color='crimson')
axes[0].set_xlabel('Trade Number', fontsize=12)
axes[0].set_ylabel('Cumulative Regret (Missed Wins)', fontsize=12)
axes[0].set_title('A/B Test Waste: Regret Grows Linearly', fontsize=14, fontweight='bold')
axes[0].grid(alpha=0.3)

# Add annotation
final_regret = cumulative_regret[-1]
axes[0].annotate(f'Total waste: {final_regret:.0f} missed wins',
                xy=(n_trades, final_regret), xytext=(n_trades*0.5, final_regret*0.8),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5),
                fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# Right plot: P-value over time
if p_values:
    trade_nums, p_vals = zip(*p_values)
    axes[1].plot(trade_nums, p_vals, linewidth=2, color='steelblue', marker='o', markersize=3)
    axes[1].axhline(y=0.05, color='red', linestyle='--', linewidth=1.5, label='α = 0.05')
    axes[1].set_xlabel('Trade Number', fontsize=12)
    axes[1].set_ylabel('P-Value', fontsize=12)
    axes[1].set_title('When Could We Have Stopped?', fontsize=14, fontweight='bold')
    axes[1].set_yscale('log')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    # Find when significance was first reached
    significant_at = None
    for t, p in p_values:
        if p < 0.05:
            significant_at = t
            break
    
    if significant_at:
        axes[1].axvline(x=significant_at, color='green', linestyle=':', linewidth=2, alpha=0.7)
        axes[1].text(significant_at, 0.5, f'  Significant at\n  trade {significant_at}',
                    fontsize=10, color='green', va='top')

plt.tight_layout()
plt.show()

if significant_at:
    wasted_trades = n_trades - significant_at
    wasted_regret = cumulative_regret[-1] - cumulative_regret[significant_at]
    print(f"\n🚨 Key Insight:")
    print(f"  - Significance reached at trade {significant_at}")
    print(f"  - But we continued for {wasted_trades:,} more trades")
    print(f"  - Accumulating {wasted_regret:.1f} additional regret")
    print(f"  - That's {wasted_regret/wasted_trades:.1%} waste per trade")

## The Problem: Linear Regret Growth

**What you just saw:**
- The A/B test kept allocating 50% of trades to Strategy A (5% win rate) and 50% to Strategy B (8% win rate)
- Even after statistical significance was reached (~2,000 trades), the test continued
- Every trade to Strategy A cost ~3% expected win rate
- Total regret: ~150 missed wins over 10,000 trades

**Why this happens:**
- A/B tests prioritize **certainty** over **efficiency**
- Fixed allocation means you keep betting on the loser to gather more evidence
- In trading, this means allocating capital to underperforming strategies

**What bandits do differently:**
- Start with 50/50 exploration
- Quickly shift allocation toward the winner (e.g., 80/20, then 90/10)
- Continue occasional exploration to detect if the loser improves
- Result: **Logarithmic regret** instead of linear

## Modify This: Change the Parameters

Try adjusting these parameters to see how regret changes:

1. **Bigger gap:** Set `strategy_A_win_rate = 0.03` and `strategy_B_win_rate = 0.10`
   - Hypothesis: Larger gap = faster significance, but also more regret per trade to the loser

2. **Smaller gap:** Set both to `0.070` and `0.075` (only 0.5% difference)
   - Hypothesis: Takes longer to reach significance, but regret per trade is smaller

3. **More trades:** Change `n_trades = 50000`
   - Hypothesis: Regret keeps growing linearly — no diminishing waste

**Re-run cells 2-4 to see the effect.**

In [ ]:
# YOUR TURN: Modify parameters and re-run

# Try these variations:
# strategy_A_win_rate = 0.03  # Much worse
# strategy_B_win_rate = 0.10  # Much better

# strategy_A_win_rate = 0.070  # Very close
# strategy_B_win_rate = 0.075

# n_trades = 50000  # Longer horizon

print("Modify the parameters above, then re-run from cell 2!")

## Real-World Implications for Commodity Trading

### Scenario: Testing Two Inventory Strategies

You're a refined products trader testing two strategies for crude oil inventory decisions:

- **Strategy A:** Traditional calendar spread signals (momentum-based)
- **Strategy B:** Machine learning model predicting refinery margins (mean-reversion-ish)

**A/B test approach:**
- Allocate $1M to each strategy for 6 months
- Strategy A returns 5%, Strategy B returns 8%
- After 3 months, B is clearly better (p < 0.01), but you continue
- **Cost:** For 3 months, you allocated $1M to the worse strategy
- **Opportunity cost:** ~$30K (3% difference × $1M)

**Bandit approach:**
- Start 50/50 in month 1
- Shift to 70/30 favoring B in month 2 (early evidence)
- Shift to 85/15 in month 3 (strong evidence)
- Maintain 95/5 in months 4-6 (exploit with light exploration)
- **Cost:** Much lower — mostly betting on the winner
- **Savings:** ~$20K compared to A/B test

### When A/B Tests Still Make Sense

- **Regulatory requirements:** Need p-values for compliance (e.g., proving a risk model)
- **One-time decisions:** Launching a new trading desk (can't adapt mid-launch)
- **Stakeholder buy-in:** Management demands "statistical proof" before committing

But for **ongoing operational decisions** (which sector to trade, which signal to use, which contract to prefer), bandits are superior.

## Summary

**What you learned:**
1. A/B tests waste traffic on inferior options due to fixed 50/50 allocation
2. Regret grows **linearly** with the number of trials
3. Even after reaching statistical significance, A/B tests continue wasting resources
4. In commodity trading, this translates to allocating capital to underperforming strategies

**What's next:**
- **Notebook 02:** Explore-exploit tradeoff with interactive controls
- **Notebook 03:** Apply these concepts to real commodity price data
- **Module 1:** Learn epsilon-greedy, the simplest adaptive bandit algorithm

**Key takeaway:** A/B testing is a snapshot; bandits are a steering wheel. When you're trading live capital, you need to learn AND earn simultaneously.